## **2. Feature Engineering**

### Token Level Features
&emsp;a) Imports & Load Data

In [32]:
import pandas as pd
import numpy as np

In [33]:
df = pd.read_csv('data/fr_en_cleaned_train.csv')
df.head(5)

,user_id,sent_id,token,POS,Morpho-Syntactic_Features,Dependency-Relation,Dependancy-Head,time,p_recall
0,YjS/mQOx,8XTyQUAl01,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,det,2,14.0,0
1,YjS/mQOx,8XTyQUAl01,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,ROOT,0,14.0,0
2,YjS/mQOx,8XTyQUAl02,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,nsubj,4,14.0,0
3,YjS/mQOx,8XTyQUAl02,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,cop,4,14.0,0
4,YjS/mQOx,8XTyQUAl02,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,det,4,14.0,0


In [34]:
tokens = df[['token', 'POS', 'Morpho-Syntactic_Features']]
tokens = tokens.drop_duplicates()
tokens.head(10)
print(tokens['token'].is_unique)

duplicate_rows = tokens[tokens.duplicated(subset=['token'], keep=False)]
print(duplicate_rows[duplicate_rows['token'] == 'suis'])
# i am preserving dup tokens because they have different POS and morpho-syntactic features which could be important for the model to learn. For example "suis" can be a verb (first person singular of "être") or a noun (plural of "sui" which is a type of plant). Removing duplicates would result in loss of information that could be relevant for the model to learn the correct associations between tokens, their POS tags, and their morpho-syntactic features.

False
      token   POS                          Morpho-Syntactic_Features
3      suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...
178    suis   AUX  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...
7504   suis  VERB  Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbF...
40832  suis  VERB  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...
50836  suis   AUX  Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...
74901  suis  VERB  Mood=Sub|Number=Plur|Person=2|Tense=Pres|VerbF...


&emsp;b) Supplment Token Level Data
 * Specifically, I am using the Lexique data base to add information about syllable count, general word frequency to give allusion to how complex or uncommon the word is

In [35]:
lexique = pd.read_csv('data/OpenLexicon.tsv', sep='\t')
lexique = lexique.rename(columns={'ortho': 'token', 'Lexique4__Lemme' : 'lemma','Lexique4__SyllNb' : 'syllable_count'})
lexique.head(5)

,token,lemma,Lexique4__Cgram,Lexique4__CgramOrtho,Lexique4__FreqOrtho,syllable_count
0,a,a,NOM,"NOM,AUX,VER",11684.443,1
1,a,avoir,VER,"NOM,AUX,VER",11684.443,1
2,a,avoir,AUX,"NOM,AUX,VER",11684.443,1
3,a capella,a capella,ADV,ADV,0.244,4
4,a cappella,a cappella,ADV,ADV,0.241,4


 * The frequency column is how many times a word occurs out of a million instances. This is heavily skewed with a range of 11000+ to smaller that 0.3. For this reason I have decided to normalize the with column with a log transformation.

In [36]:
lexique['ortho_freq'] = np.log1p(lexique['Lexique4__FreqOrtho'])
tokens = pd.merge(
    tokens.assign(token_lower=tokens['token'].str.lower()),
    lexique[['token', 'syllable_count', 'ortho_freq']].rename(columns={'token': 'token_lower'}),
    on='token_lower',
    how='left'
).drop(columns='token_lower').drop_duplicates()

tokens.head(10)



,token,POS,Morpho-Syntactic_Features,syllable_count,ortho_freq
0,Le,DET,Definite=Def|Gender=Masc|Number=Sing|fPOS=DET++,1.0,9.713540
3,garçon,NOUN,Gender=Masc|Number=Sing|fPOS=NOUN++,2.0,5.244537
4,Je,PRON,Number=Sing|Person=1|PronType=Prs|fPOS=PRON++,1.0,10.163428
6,suis,VERB,Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbF...,1.0,8.330842
10,une,DET,Definite=Ind|Gender=Fem|Number=Sing|PronType=D...,1.0,9.039167
14,femme,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,1.0,6.585643
15,La,DET,Definite=Def|Gender=Fem|Number=Sing|fPOS=DET++,1.0,9.613319
19,fille,NOUN,Gender=Fem|Number=Sing|fPOS=NOUN++,1.0,6.447431
20,un,DET,Definite=Ind|Gender=Masc|Number=Sing|PronType=...,1.0,9.437757
24,riche,ADJ,Gender=Fem|Number=Sing|fPOS=ADJ++,1.0,3.956097
